In [ ]:
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from copy import copy
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

In [16]:

DATA_DIR = Path(
    "dataset/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data"
)

print("Folder de lucru:", Path.cwd())
print("Folder date:", DATA_DIR.resolve())

required = [
    "images",
    "variants.txt",
    "images_variant_train.txt",
    "images_variant_val.txt",
    "images_variant_test.txt",
]

for name in required:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Lipsește: {path}")

print("Structura necesară există.")

Folder de lucru: /home/mario/CV3
Folder date: /home/mario/CV3/dataset/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data
Structura necesară există.


In [17]:
class AircraftDataset(Dataset):
    def __init__(self, data_dir, split, transform=None):
        if split not in ("train", "val", "test"):
            raise ValueError("split trebuie să fie train, val sau test")

        self.data_dir = Path(data_dir)
        self.transform = transform

        # Aceeași ordine a claselor pentru toate split-urile
        self.classes = [
            line.strip()
            for line in (self.data_dir / "variants.txt")
            .read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

        self.class_to_idx = {
            name: index for index, name in enumerate(self.classes)
        }

        annotation_file = (
            self.data_dir / f"images_variant_{split}.txt"
        )

        self.samples = []

        for line in annotation_file.read_text(
            encoding="utf-8"
        ).splitlines():
            if not line.strip():
                continue

            # Numele clasei poate conține spații
            image_id, class_name = line.strip().split(maxsplit=1)

            image_path = self.data_dir / "images" / f"{image_id}.jpg"
            label = self.class_to_idx[class_name]

            if not image_path.is_file():
                raise FileNotFoundError(f"Imagine lipsă: {image_path}")

            self.samples.append((image_path, label))

        if not self.samples:
            raise ValueError(f"Split-ul {split} este gol")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]

        with Image.open(image_path) as img:
            image = img.convert("RGB")

        # Eliminarea benzii inferioare de copyright
        width, height = image.size
        if height <= 20:
            raise ValueError(f"Imagine prea mică: {image_path}")

        image = image.crop((0, 0, width, height - 20))

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


all_samples = []

for split in ("train", "val", "test"):
    dataset = AircraftDataset(DATA_DIR, split)
    all_samples.extend(dataset.samples)

# Verificăm că fiecare imagine apare o singură dată
paths = [path for path, _ in all_samples]
assert len(paths) == len(set(paths)), "Există imagini duplicate!"

all_labels = [label for _, label in all_samples]

# 60% train, 40% temporar
train_samples, remaining_samples = train_test_split(
    all_samples,
    test_size=0.40,
    stratify=all_labels,
    random_state=42,
)

# Împărțim cei 40% în două: 20% val și 20% test
remaining_labels = [label for _, label in remaining_samples]

val_samples, test_samples = train_test_split(
    remaining_samples,
    test_size=0.50,
    stratify=remaining_labels,
    random_state=42,
)

# Dataset-uri separate, fiecare cu transformările potrivite
def make_dataset(samples, transform):
    result = copy(dataset)
    result.samples = list(samples)
    result.transform = transform
    return result

train_dataset = make_dataset(train_samples, train_transform)
val_dataset = make_dataset(val_samples, eval_transform)
test_dataset = make_dataset(test_samples, eval_transform)

NUM_CLASSES = len(train_dataset.classes)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "pin_memory": torch.cuda.is_available(),
}

train_loader = DataLoader(
    train_dataset, shuffle=True, **loader_options
)
val_loader = DataLoader(
    val_dataset, shuffle=False, **loader_options
)
test_loader = DataLoader(
    test_dataset, shuffle=False, **loader_options
)

total = len(all_samples)

for name, ds in [
    ("train", train_dataset),
    ("val", val_dataset),
    ("test", test_dataset),
]:
    print(
        f"{name}: {len(ds)} imagini "
        f"({100 * len(ds) / total:.1f}%)"
    )

train: 6000 imagini (60.0%)
val: 2000 imagini (20.0%)
test: 2000 imagini (20.0%)


In [19]:
datasets = {
    "train": train_dataset,
    "val": val_dataset,
    "test": test_dataset,
}

loaders = {
    "train": train_loader,
    "val": val_loader,
    "test": test_loader,
}

total = sum(len(ds) for ds in datasets.values())

for name, ds in datasets.items():
    counts = Counter(label for _, label in ds.samples)

    assert ds.classes == train_dataset.classes
    assert set(counts) == set(range(NUM_CLASSES))

    print(
        f"{name:5s}: {len(ds):5d} imagini "
        f"({100 * len(ds) / total:.2f}%) | "
        f"{len(counts)} clase | "
        f"{min(counts.values())}–{max(counts.values())} imagini/clasă"
    )

image_sets = {
    name: {path.name for path, _ in ds.samples}
    for name, ds in datasets.items()
}

for name, ds in datasets.items():
    assert len(image_sets[name]) == len(ds), (
        f"Imagini duplicate în {name}"
    )

for first, second in [
    ("train", "val"),
    ("train", "test"),
    ("val", "test"),
]:
    assert image_sets[first].isdisjoint(image_sets[second]), (
        f"Suprapunere între {first} și {second}"
    )

for name, loader in loaders.items():
    images, labels = next(iter(loader))

    assert images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
    assert images.dtype == torch.float32
    assert torch.isfinite(images).all()
    assert labels.dtype == torch.int64
    assert labels.min().item() >= 0
    assert labels.max().item() < NUM_CLASSES

    print(
        f"{name}: imagini {tuple(images.shape)}, "
        f"etichete {tuple(labels.shape)}"
    )

print("\nVerificările au trecut.")

train:  6000 imagini (60.00%) | 100 clase | 60–60 imagini/clasă
val  :  2000 imagini (20.00%) | 100 clase | 20–20 imagini/clasă
test :  2000 imagini (20.00%) | 100 clase | 20–20 imagini/clasă
train: imagini (32, 3, 224, 224), etichete (32,)
val: imagini (32, 3, 224, 224), etichete (32,)
test: imagini (32, 3, 224, 224), etichete (32,)

Verificările au trecut.
